In [ ]:
import torch
import torch.nn as nn
import random
import json

# Load English-Tamil word pairs from demo_pairs.json (100 pairs for faster training)
print("Loading data from demo_pairs.json...")
with open('demo_pairs.json', 'r', encoding='utf-8') as f:
    pairs = json.load(f)

print(f"Loaded {len(pairs)} English-Tamil pairs")
print("\nFirst 10 examples:")
for i, (eng, tam) in enumerate(pairs[:10], 1):
    print(f"{i:2}. {eng:20} -> {tam}")

# We need special tokens to tell the model when to Start <SOS> and End <EOS>
SOS_token = 0
EOS_token = 1

# Create Character-to-Index mappings
eng_chars = set("".join([p[0] for p in pairs]))
tam_chars = set("".join([p[1] for p in pairs]))

eng2idx = {char: i+2 for i, char in enumerate(eng_chars)}
eng2idx['<SOS>'] = SOS_token
eng2idx['<EOS>'] = EOS_token

tam2idx = {char: i+2 for i, char in enumerate(tam_chars)}
tam2idx['<SOS>'] = SOS_token
tam2idx['<EOS>'] = EOS_token
idx2tam = {i: char for char, i in tam2idx.items()}

print(f"\nEnglish vocabulary size: {len(eng2idx)} characters")
print(f"Tamil vocabulary size: {len(tam2idx)} characters")

# Helper function to convert a word into a tensor of numbers
def word_to_tensor(word, lang_dict):
    indexes = [lang_dict[char] for char in word]
    indexes.append(EOS_token) # Always end with <EOS>
    return torch.tensor(indexes, dtype=torch.long).view(1, -1) # Shape: (1, Sequence Length)

Loading data from demo_pairs.json...
Loaded 97 English-Tamil pairs

First 10 examples:
 1. About                -> பற்றி
 2. Accessories          -> பயன்பாடுகள்
 3. Accountant           -> கணக்காளர்
 4. Acid                 -> அமிலம்
 5. Acknowledgement      -> ஒப்புரை
 6. Acquisition          -> கையகப்படுத்தல்
 7. across               -> கடந்து
 8. Additional           -> கூடுதல்
 9. Adhesives            -> பசைமங்கள்
10. Adjective            -> பெயரடை

English vocabulary size: 32 characters
Tamil vocabulary size: 43 characters


In [ ]:
HIDDEN_SIZE = 64

class CharEncoder(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, HIDDEN_SIZE)
        self.lstm = nn.LSTM(HIDDEN_SIZE, HIDDEN_SIZE, batch_first=True)

    def forward(self, x):
        embedded = self.embedding(x)
        _, (hidden, cell) = self.lstm(embedded)
        return hidden, cell # We only keep the final memory

In [ ]:
class CharDecoder(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, HIDDEN_SIZE)
        self.lstm = nn.LSTM(HIDDEN_SIZE, HIDDEN_SIZE, batch_first=True)
        self.fc = nn.Linear(HIDDEN_SIZE, vocab_size)

    def forward(self, x, hidden, cell):
        embedded = self.embedding(x)
        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        prediction = self.fc(output)
        return prediction, hidden, cell

In [ ]:
encoder = CharEncoder(len(eng2idx))
decoder = CharDecoder(len(tam2idx))

# Optimizers to update the weights
enc_optimizer = torch.optim.Adam(encoder.parameters(), lr=0.005)
dec_optimizer = torch.optim.Adam(decoder.parameters(), lr=0.005)
criterion = nn.CrossEntropyLoss()

print("Training the Character-Level Translator...")
print(f"Dataset size: {len(pairs)} pairs")
print(f"Training for 100 epochs...\n")

for epoch in range(100):
    total_loss = 0

    for eng_word, tam_word in pairs:
        enc_optimizer.zero_grad()
        dec_optimizer.zero_grad()

        # 1. Prepare inputs
        input_tensor = word_to_tensor(eng_word, eng2idx)
        target_tensor = word_to_tensor(tam_word, tam2idx)

        # 2. Encoder reads English
        hidden, cell = encoder(input_tensor)

        # 3. Decoder writes Tamil (Starts with <SOS>)
        decoder_input = torch.tensor([[SOS_token]], dtype=torch.long)
        loss = 0

        # Step through the target word character by character
        for t in range(target_tensor.size(1)):
            output, hidden, cell = decoder(decoder_input, hidden, cell)
            target_char = target_tensor[:, t]

            loss += criterion(output.view(1, -1), target_char)
            decoder_input = target_char.unsqueeze(1) # Teacher forcing

        loss.backward()
        enc_optimizer.step()
        dec_optimizer.step()
        total_loss += loss.item()

    if epoch % 25 == 0:
        avg_loss = total_loss/len(pairs)
        print(f"Epoch {epoch:3d} | Average Loss: {avg_loss:.4f}")

print("\nTraining completed!")

Training the Character-Level Translator...
Dataset size: 97 pairs
Training for 100 epochs...

Epoch   0 | Average Loss: 32.7463
Epoch  25 | Average Loss: 0.2103
Epoch  50 | Average Loss: 0.0301
Epoch  75 | Average Loss: 0.1018

Training completed!


In [ ]:
def translate(word):
    with torch.no_grad():
        # 1. Read the English word
        input_tensor = word_to_tensor(word, eng2idx)
        hidden, cell = encoder(input_tensor)

        # 2. Start writing Tamil
        decoder_input = torch.tensor([[SOS_token]], dtype=torch.long)
        translated_chars = []

        # Generate up to 20 characters to prevent infinite loops
        for _ in range(20):
            output, hidden, cell = decoder(decoder_input, hidden, cell)

            # Find the most likely character
            top_char_idx = output.argmax(dim=2).item()

            # Stop if it generates the <EOS> token
            if top_char_idx == EOS_token:
                break

            translated_chars.append(idx2tam[top_char_idx])

            # Feed the generated character back in as the next input
            decoder_input = torch.tensor([[top_char_idx]], dtype=torch.long)

        return "".join(translated_chars)

# Test it live!
print("\n--- Live Translation Test ---")
# Test on some random samples from training data
test_samples = random.sample(pairs, 10)
for eng, actual_tamil in test_samples:
    predicted = translate(eng)
    match = "✓" if predicted == actual_tamil else "✗"
    print(f"{match} English: {eng:20} -> Predicted: {predicted:25} | Actual: {actual_tamil}")


--- Live Translation Test ---
✓ English: Advertisers          -> Predicted: விளம்பரதாரர்              | Actual: விளம்பரதாரர்
✓ English: Automobiles          -> Predicted: தானியங்கிகள்              | Actual: தானியங்கிகள் 
✓ English: Autorickshaw         -> Predicted: தானி                      | Actual: தானி
✓ English: Auto-service         -> Predicted: தானிப் பணியகம்            | Actual: தானிப் பணியகம்
✓ English: Cancel               -> Predicted: விடு                      | Actual: விடு
✓ English: Constructions        -> Predicted: கட்டுமானங்கள்             | Actual: கட்டுமானங்கள்
✓ English: Brothers             -> Predicted: உடன் பிறப்புகள்           | Actual: உடன் பிறப்புகள்
✓ English: Chart                -> Predicted: நிரல்படம்                 | Actual: நிரல்படம்
✓ English: Colours              -> Predicted: வண்ணங்கள் நிறங்கள்        | Actual: வண்ணங்கள் நிறங்கள்
✓ English: Calender             -> Predicted: நாள்காட்டி                | Actual: நாள்காட்டி


In [ ]:
# Test with unseen words (user input)
print("\n--- Test Your Own Word ---")
test_word = input("Enter an English word to translate: ")
try:
    translation = translate(test_word)
    print(f"English: {test_word} -> Tamil: {translation}")
except KeyError as e:
    print(f"Error: Character '{e.args[0]}' not in vocabulary. Try a simpler word.")


--- Test Your Own Word ---
Enter an English word to translate: book
English: book -> Tamil: சிறுகடை
